# Stage 3: PCA & MLP-Phase Baseline (B3)
**Environment:** Kaggle Notebook (GPU T4)

In [ ]:
import numpy as np, pandas as pd, os, glob, json
from typing import Dict, Tuple, List
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120; plt.rcParams['font.family'] = 'serif'; sns.set_style('whitegrid')

INPUT_DIR = '/kaggle/input/artifact-dataset'
OUTPUT_DIR = '/kaggle/working'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

CONFIG = {
    'n_pca_components': 128, 'mlp_hidden_dims': [256, 128, 64], 'dropout': 0.3,
    'learning_rate': 1e-3, 'weight_decay': 1e-4, 'batch_size': 64,
    'epochs': 50, 'early_stopping_patience': 10, 'test_size': 0.2,
    'val_size': 0.15, 'random_state': 42,
}

In [ ]:
# Cell 2: Load Phase Maps & Prepare
cache_files = sorted(glob.glob(os.path.join(CACHE_DIR, 'phase_maps_*.npy')))
if not cache_files:
    raise FileNotFoundError(f'No cache in {CACHE_DIR}. Run Stage 2 first!')
phase_results = np.load(cache_files[-1], allow_pickle=True).item()

meta_files = glob.glob(os.path.join(INPUT_DIR, '**', 'metadata.csv'), recursive=True)
metadata_df = pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf)))
                         for mf in meta_files], ignore_index=True) if meta_files else None

X_list, y_list = [], []
for gen, maps in phase_results.items():
    if metadata_df is not None:
        gen_df = metadata_df[metadata_df['generator'] == gen]
        is_real = len(gen_df) > 0 and gen_df['target'].mode().iloc[0] == 0
    else:
        is_real = 'real' in gen.lower()
    label = 0 if is_real else 1
    for m in maps:
        X_list.append(m.flatten())
        y_list.append(label)

X_raw = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
print(f'Features: {X_raw.shape}, Real: {(y==0).sum()}, Fake: {(y==1).sum()}')

In [ ]:
# Cell 3: PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
n_max = min(X_scaled.shape[0], X_scaled.shape[1], 500)
pca_full = PCA(n_components=n_max, random_state=42)
pca_full.fit(X_scaled)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PCA Variance Analysis', fontsize=14, fontweight='bold')
ax1.bar(range(1, min(51, n_max+1)), pca_full.explained_variance_ratio_[:50], color='steelblue')
ax1.set_xlabel('Component'); ax1.set_ylabel('Variance Ratio'); ax1.set_title('Individual (Top 50)')
ax2.plot(range(1, len(cumulative_var)+1), cumulative_var, color='steelblue', lw=2)
ax2.axhline(y=0.95, color='red', ls='--', label='95%')
n_comp = CONFIG['n_pca_components']
if n_comp <= len(cumulative_var):
    ax2.axvline(x=n_comp, color='green', ls=':', label=f'n={n_comp} ({cumulative_var[n_comp-1]:.1%})')
ax2.set_xlabel('Components'); ax2.set_ylabel('Cumulative Variance'); ax2.legend(); ax2.set_xlim(0, min(300, n_max))
plt.tight_layout(); plt.show()

pca = PCA(n_components=n_comp, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f'PCA: {X_pca.shape}, variance retained: {pca.explained_variance_ratio_.sum():.3f}')
np.save(os.path.join(MODEL_DIR, 'pca_components.npy'), pca.components_)
np.save(os.path.join(MODEL_DIR, 'scaler_mean.npy'), scaler.mean_)

In [ ]:
# Cell 4: MLP-Phase Model & Training
class MLPPhase(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(True), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x): return self.network(x).squeeze(-1)

X_tv, X_test, y_tv, y_test = train_test_split(X_pca, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=42)
print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)), batch_size=64)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test), torch.LongTensor(y_test)), batch_size=64)

model = MLPPhase(n_comp, CONFIG['mlp_hidden_dims'], CONFIG['dropout']).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'MLP params: {n_params:,}')

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
history = {'train_loss':[], 'val_loss':[], 'train_auc':[], 'val_auc':[]}
best_auc, patience_ctr = 0.0, 0

for epoch in tqdm(range(CONFIG['epochs']), desc='MLP Training'):
    model.train(); t_loss, t_preds, t_tgts = 0, [], []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.float().to(DEVICE)
        optimizer.zero_grad(); logits = model(X_b); loss = criterion(logits, y_b)
        loss.backward(); optimizer.step()
        t_loss += loss.item()*len(y_b); t_preds.append(torch.sigmoid(logits).detach().cpu().numpy()); t_tgts.append(y_b.cpu().numpy())
    t_preds, t_tgts = np.concatenate(t_preds), np.concatenate(t_tgts)
    t_auc = roc_auc_score(t_tgts, t_preds) if len(np.unique(t_tgts))>1 else 0
    model.eval(); v_loss, v_preds, v_tgts = 0, [], []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.float().to(DEVICE)
            logits = model(X_b); v_loss += criterion(logits, y_b).item()*len(y_b)
            v_preds.append(torch.sigmoid(logits).cpu().numpy()); v_tgts.append(y_b.cpu().numpy())
    v_preds, v_tgts = np.concatenate(v_preds), np.concatenate(v_tgts)
    v_auc = roc_auc_score(v_tgts, v_preds) if len(np.unique(v_tgts))>1 else 0
    scheduler.step(v_auc)
    history['train_loss'].append(t_loss/len(train_loader.dataset))
    history['val_loss'].append(v_loss/len(val_loader.dataset))
    history['train_auc'].append(t_auc); history['val_auc'].append(v_auc)
    if v_auc > best_auc:
        best_auc = v_auc; patience_ctr = 0
        torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'mlp_phase_b3_best.pth'))
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG['early_stopping_patience']:
            print(f'Early stop at epoch {epoch+1}'); break

model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'mlp_phase_b3_best.pth'), weights_only=True))

In [ ]:
# Cell 5: Evaluation
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ep = range(1, len(history['train_loss'])+1)
axes[0].plot(ep, history['train_loss'], label='Train'); axes[0].plot(ep, history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(ep, history['train_auc'], label='Train'); axes[1].plot(ep, history['val_auc'], label='Val')
axes[1].set_title('AUC'); axes[1].legend()
plt.tight_layout(); plt.show()

model.eval(); all_p, all_t = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        all_p.append(torch.sigmoid(model(X_b.to(DEVICE))).cpu().numpy()); all_t.append(y_b.numpy())
test_preds, test_targets = np.concatenate(all_p), np.concatenate(all_t)
test_acc = accuracy_score(test_targets, (test_preds>0.5).astype(int))
test_auc = roc_auc_score(test_targets, test_preds)
print(f'MLP-Phase (B3): Accuracy={test_acc:.4f}, AUC={test_auc:.4f}, Params={n_params:,}')
print(classification_report(test_targets, (test_preds>0.5).astype(int), target_names=['Real','Fake']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fpr, tpr, _ = roc_curve(test_targets, test_preds)
ax1.plot(fpr, tpr, lw=2, label=f'AUC={test_auc:.3f}'); ax1.plot([0,1],[0,1],'k--',alpha=0.3)
ax1.set_title('ROC'); ax1.legend()
cm = confusion_matrix(test_targets, (test_preds>0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2, xticklabels=['Real','Fake'], yticklabels=['Real','Fake'])
ax2.set_title('Confusion Matrix'); plt.tight_layout(); plt.show()

with open(os.path.join(MODEL_DIR, 'results_b3.json'), 'w') as f:
    json.dump({'model':'MLP-Phase (B3)','test_accuracy':float(test_acc),'test_auc':float(test_auc),'n_parameters':n_params}, f, indent=2)
print('B3 results saved.')